
# Day 1 — Section 2: Log Probabilities

This section connects the distribution an LLM computes at each generation step
to the information exposed by an inference API. You will request token-level
log probabilities, inspect the response structure, and reason about how that
extra observability changes an attacker's options.

## Table of Contents

- [Content & Learning Objectives](#content--learning-objectives)
- [Logprobs: What the Model Actually Computes](#logprobs-what-the-model-actually-computes)
    - [Exercise 1.2.1: Logprobs](#exercise-121-logprobs)

## Content & Learning Objectives

> **Learning Objectives**
> - Request and parse token-level log probabilities from a chat API
> - Relate logits, log probabilities, probabilities, and token rankings
> - Explain how output-distribution access can support extraction and adversarial optimization


In [3]:


# %%
import json
import math
import os
import sys
from collections.abc import Callable
from pathlib import Path

from openai import OpenAI
from openai.types.chat import ChatCompletionMessageParam

__file__ = "/Users/nl/aisb/aisb/1.2-logprobs"
_root = next(p for p in Path(__file__).resolve().parents if (p / "aisb_utils").is_dir())
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from aisb_utils import report
from aisb_utils.env import load_dotenv

load_dotenv()

# OpenRouter client
openrouter_client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY", ""),
)

## Logprobs: What the Model Actually Computes

An LLM produces a **probability distribution over tokens** at each step. The API can return these as `logprobs`. This is the raw output before sampling, and it reveals information the final text doesn't.

### Exercise 1.2.1: Logprobs

> **Difficulty**: 2/5
> **Importance**: 3/5

Use the [completions](https://developers.openai.com/api/reference/resources/completions/methods/create) API to make a request with `logprobs=True` and examine what comes back.


In [40]:


LOGPROBS_MODEL = "openai/gpt-4.1-mini"  # Not all models support logprobs


def get_completion_with_logprobs(
    prompt: str,
    model: str = LOGPROBS_MODEL,
    max_tokens: int = 50,
    top_logprobs: int = 5,
) -> list[list[tuple[str, float]]]:
    """Get a completion with logprobs from the API.

    Returns for each generated token a list of (token, logprob) pairs
    for the top alternatives at that position.
    """
    messages: list[ChatCompletionMessageParam] = [
        {"role": "user", "content": prompt}
    ]

    out = openrouter_client.chat.completions.create(messages=messages, model=model, logprobs=True, top_logprobs=top_logprobs, max_tokens=max_tokens)

    ret = []
    for lp in out.choices[0].logprobs.content:
        toplp_arr = []
        for toplp in lp.top_logprobs:
            toplp_arr.append( (toplp.token, toplp.logprob) )

        ret.append(toplp_arr)

    return ret


# Get logprobs for a simple prompt
token_pairs = get_completion_with_logprobs("text:Scott Galloway on GLP-1: They're now saying that it reduces the likelihood of stage one cancer going to metastatic cancer... by 50%... Everyone at this table is going to be on a GLP-1 within 24 months.\nRespond in one work what is the sentiment: ")
print(token_pairs)
completion = "".join(alts[0][0] for alts in token_pairs)
print(f"Completion: {completion}")
for alts in token_pairs[:10]:
    alt_str = ", ".join(f"{tok}({math.exp(lp):.1%})" for tok, lp in alts[:3])
    print(f"  - {alt_str}")
from section2_test import test_get_completion_with_logprobs


test_get_completion_with_logprobs(get_completion_with_logprobs)




[[('Optim', -0.1602388471364975), ('Positive', -1.9102388620376587), ('optim', -12.035239219665527), ('Hope', -12.160239219665527), (' optimistic', -14.035239219665527)], [('istic', 0.0), ('ism', -27.375), ('isit', -28.0), ('ISTIC', -29.125), ('ous', -31.375)]]
Completion: Optimistic
  - Optim(85.2%), Positive(14.8%), optim(0.0%)
  - istic(100.0%), ism(0.0%), isit(0.0%)
  All tests passed!
section2_test.test_get_completion_with_logprobs passed.


In [ ]:
ChatCompletion(id='gen-1785704642-wGjlNlfKTXwHYMVBGKPA', choices=[Choice(finish_reason='stop', index=0, logprobs=ChoiceLogprobs(content=[ChatCompletionTokenLogprob(token="I'd", bytes=[73, 39, 100], logprob=-0.8311349153518677, top_logprobs=[TopLogprob(token='I', bytes=[73], logprob=-0.5811349153518677), TopLogprob(token="I'd", bytes=[73, 39, 100], logprob=-0.8311349153518677), TopLogprob(token='Great', bytes=[71, 114, 101, 97, 116], logprob=-5.706134796142578), TopLogprob(token="That's", bytes=[84, 104, 97, 116, 39, 115], logprob=-7.081134796142578), TopLogprob(token='That', bytes=[84, 104, 97, 116], logprob=-7.331134796142578)]), ChatCompletionTokenLogprob(token=' love', bytes=[32, 108, 111, 118, 101], logprob=-1.9361264946837764e-07, top_logprobs=[TopLogprob(token=' love', bytes=[32, 108, 111, 118, 101], logprob=-1.9361264946837764e-07), TopLogprob(token=' be', bytes=[32, 98, 101], logprob=-15.625), TopLogprob(token='love', bytes=[108, 111, 118, 101], logprob=-19.25), TopLogprob(token=' like', bytes=[32, 108, 105, 107, 101], logprob=-20.0), TopLogprob(token=' LOVE', bytes=[32, 76, 79, 86, 69], logprob=-21.125)]), ChatCompletionTokenLogprob(token=' to', bytes=[32, 116, 111], logprob=0.0, top_logprobs=[TopLogprob(token=' to', bytes=[32, 116, 111], logprob=0.0), TopLogprob(token=' that', bytes=[32, 116, 104, 97, 116], logprob=-19.5), TopLogprob(token=' hearing', bytes=[32, 104, 101, 97, 114, 105, 110, 103], logprob=-19.5), TopLogprob(token=' for', bytes=[32, 102, 111, 114], logprob=-20.375), TopLogprob(token=' it', bytes=[32, 105, 116], logprob=-20.625)]), ChatCompletionTokenLogprob(token=' hear', bytes=[32, 104, 101, 97, 114], logprob=-2.45848218582978e-06, top_logprobs=[TopLogprob(token=' hear', bytes=[32, 104, 101, 97, 114], logprob=-2.45848218582978e-06), TopLogprob(token=' see', bytes=[32, 115, 101, 101], logprob=-13.12500286102295), TopLogprob(token=' know', bytes=[32, 107, 110, 111, 119], logprob=-15.50000286102295), TopLogprob(token='hear', bytes=[104, 101, 97, 114], logprob=-16.125001907348633), TopLogprob(token=' read', bytes=[32, 114, 101, 97, 100], logprob=-18.250001907348633)]), ChatCompletionTokenLogprob(token=' your', bytes=[32, 121, 111, 117, 114], logprob=-0.10020710527896881, top_logprobs=[TopLogprob(token=' your', bytes=[32, 121, 111, 117, 114], logprob=-0.10020710527896881), TopLogprob(token=' it', bytes=[32, 105, 116], logprob=-2.3502070903778076), TopLogprob(token=' or', bytes=[32, 111, 114], logprob=-15.100207328796387), TopLogprob(token=' about', bytes=[32, 97, 98, 111, 117, 116], logprob=-17.22520637512207), TopLogprob(token=' what', bytes=[32, 119, 104, 97, 116], logprob=-17.35020637512207)]), ChatCompletionTokenLogprob(token=' favorite', bytes=[32, 102, 97, 118, 111, 114, 105, 116, 101], logprob=-3.547789674485102e-05, top_logprobs=[TopLogprob(token=' favorite', bytes=[32, 102, 97, 118, 111, 114, 105, 116, 101], logprob=-3.547789674485102e-05), TopLogprob(token=' joke', bytes=[32, 106, 111, 107, 101], logprob=-10.250035285949707), TopLogprob(token=' favourite', bytes=[32, 102, 97, 118, 111, 117, 114, 105, 116, 101], logprob=-18.750036239624023), TopLogprob(token='favorite', bytes=[102, 97, 118, 111, 114, 105, 116, 101], logprob=-20.875036239624023), TopLogprob(token=' funny', bytes=[32, 102, 117, 110, 110, 121], logprob=-22.000036239624023)]), ChatCompletionTokenLogprob(token=' joke', bytes=[32, 106, 111, 107, 101], logprob=-4.320199877838604e-07, top_logprobs=[TopLogprob(token=' joke', bytes=[32, 106, 111, 107, 101], logprob=-4.320199877838604e-07), TopLogprob(token='!', bytes=[33], logprob=-15.0), TopLogprob(token=',', bytes=[44], logprob=-20.0), TopLogprob(token=' one', bytes=[32, 111, 110, 101], logprob=-21.375), TopLogprob(token=' ', bytes=[32], logprob=-21.625)]), ChatCompletionTokenLogprob(token='!', bytes=[33], logprob=-3.531315314830863e-06, top_logprobs=[TopLogprob(token='!', bytes=[33], logprob=-3.531315314830863e-06), TopLogprob(token='—', bytes=[226, 128, 148], logprob=-12.625003814697266), TopLogprob(token=' if', bytes=[32, 105, 102], logprob=-16.375003814697266), TopLogprob(token=' whenever', bytes=[32, 119, 104, 101, 110, 101, 118, 101, 114], logprob=-18.000003814697266), TopLogprob(token=' —', bytes=[32, 226, 128, 148], logprob=-18.750003814697266)]), ChatCompletionTokenLogprob(token=' Please', bytes=[32, 80, 108, 101, 97, 115, 101], logprob=-0.05415542423725128, top_logprobs=[TopLogprob(token=' Please', bytes=[32, 80, 108, 101, 97, 115, 101], logprob=-0.05415542423725128), TopLogprob(token=' What', bytes=[32, 87, 104, 97, 116], logprob=-3.6791553497314453), TopLogprob(token=' Go', bytes=[32, 71, 111], logprob=-4.054155349731445), TopLogprob(token=' Feel', bytes=[32, 70, 101, 101, 108], logprob=-5.679155349731445), TopLogprob(token=' Could', bytes=[32, 67, 111, 117, 108, 100], logprob=-5.929155349731445)]), ChatCompletionTokenLogprob(token=' share', bytes=[32, 115, 104, 97, 114, 101], logprob=-0.062219977378845215, top_logprobs=[TopLogprob(token=' share', bytes=[32, 115, 104, 97, 114, 101], logprob=-0.062219977378845215), TopLogprob(token=' go', bytes=[32, 103, 111], logprob=-2.8122200965881348), TopLogprob(token=' tell', bytes=[32, 116, 101, 108, 108], logprob=-8.312219619750977), TopLogprob(token=' feel', bytes=[32, 102, 101, 101, 108], logprob=-12.437219619750977), TopLogprob(token=',', bytes=[44], logprob=-12.812219619750977)]), ChatCompletionTokenLogprob(token=' it', bytes=[32, 105, 116], logprob=0.0, top_logprobs=[TopLogprob(token=' it', bytes=[32, 105, 116], logprob=0.0), TopLogprob(token='!', bytes=[33], logprob=-17.875), TopLogprob(token='—it', bytes=[226, 128, 148, 105, 116], logprob=-19.625), TopLogprob(token=',', bytes=[44], logprob=-21.0), TopLogprob(token='—', bytes=[226, 128, 148], logprob=-24.625)]), ChatCompletionTokenLogprob(token=' with', bytes=[32, 119, 105, 116, 104], logprob=-0.001061867456883192, top_logprobs=[TopLogprob(token=' with', bytes=[32, 119, 105, 116, 104], logprob=-0.001061867456883192), TopLogprob(token='!', bytes=[33], logprob=-7.001061916351318), TopLogprob(token='.', bytes=[46], logprob=-9.00106143951416), TopLogprob(token=' when', bytes=[32, 119, 104, 101, 110], logprob=-11.00106143951416), TopLogprob(token=',', bytes=[44], logprob=-11.75106143951416)]), ChatCompletionTokenLogprob(token=' me', bytes=[32, 109, 101], logprob=0.0, top_logprobs=[TopLogprob(token=' me', bytes=[32, 109, 101], logprob=0.0), TopLogprob(token=' меня', bytes=[32, 208, 188, 208, 181, 208, 189, 209, 143], logprob=-29.125), TopLogprob(token=' мне', bytes=[32, 208, 188, 208, 189, 208, 181], logprob=-30.75), TopLogprob(token='me', bytes=[109, 101], logprob=-31.375), TopLogprob(token=' ме', bytes=[32, 208, 188, 208, 181], logprob=-33.125)]), ChatCompletionTokenLogprob(token='.', bytes=[46], logprob=-0.023252815008163452, top_logprobs=[TopLogprob(token='.', bytes=[46], logprob=-0.023252815008163452), TopLogprob(token='!', bytes=[33], logprob=-3.7732527256011963), TopLogprob(token=',', bytes=[44], logprob=-12.398252487182617), TopLogprob(token='?', bytes=[63], logprob=-13.898252487182617), TopLogprob(token='—I', bytes=[226, 128, 148, 73], logprob=-14.273252487182617)])], refusal=None), message=ChatCompletionMessage(content="I'd love to hear your favorite joke! Please share it with me.", refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning=None), native_finish_reason='completed')], created=1785704642, model='openai/gpt-4.1-mini', object='chat.completion', moderation=None, service_tier='default', system_fingerprint=None, usage=CompletionUsage(completion_tokens=15, prompt_tokens=10, total_tokens=25, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=None, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=None, image_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cache_write_tokens=0, cached_tokens=0, video_tokens=0), cost=2.8e-05, is_byok=False, cost_details={'upstream_inference_cost': 2.8e-05, 'upstream_inference_prompt_cost': 4e-06, 'upstream_inference_completions_cost': 2.4e-05}), provider='OpenAI')


**Question: How can logprobs be misused by an attacker?**
<details>
<summary>Answer</summary><blockquote>

Logprobs leak information about the model's internal state beyond what sampled tokens alone reveal. The key threats include:
- **Adversarial prompt optimization**: logprobs provide a differentiable-like signal that attackers can use to iteratively refine jailbreak prompts. Instead of random guessing, they measure which token substitutions increase the probability of harmful completions, effectively using logprobs as a black-box gradient.
- **Model distillation/stealing**: logprobs expose the model's full probability distribution (or top-k), which provides far richer training signal than sampled tokens alone. An attacker can train a smaller "student" model on these soft labels, efficiently cloning the target model's behavior at a fraction of the original training cost.
 The attacker can also extract information about the model architecture and model weights via special attacks (eg,
 see [3.5-weight-extraction])

Logprobs can also aid **system prompt extraction** and **model fingerprinting** (probability distributions can identify the model version or provider). This is why some providers restrict or disable logprobs access.
</blockquote></details>